# Pairs trading

Compare four daily strategies with execution costs and rolling out-of-sample evaluation. Install `requirements.txt` first; run this notebook from the project root or the notebooks folder.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd() if (Path.cwd() / "pairs").is_dir() else Path.cwd().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pairs import (
    STRATEGIES, BacktestConfig, run_backtest, load_pair, load_universe,
    synthetic_pair, screen_pairs, ols, engle_granger, ou_params, walkforward_optimize,
)
from pairs.plots import plot_equity, plot_signals, plot_cost_sensitivity


## Data

Synthetic data is the default for reproducible offline runs. Set `USE_SYNTHETIC=False` to download market prices.

In [ ]:
USE_SYNTHETIC = True
X_T, Y_T = 'PETR3.SA', 'PETR4.SA'   

if USE_SYNTHETIC:
    df = synthetic_pair()
else:
    df = load_pair(X_T, Y_T, start='2020-01-01')
print(len(df), 'sessions', df.index[0].date(), '->', df.index[-1].date())
df.tail()

## Pair screening

Both regression directions are tested. The smaller p-value is corrected for that choice before applying Benjamini–Hochberg across pairs. Screen on a formation period that ends before any subsequent trading test.

In [ ]:
tickers = ['PETR3.SA','PETR4.SA','BBDC3.SA','BBDC4.SA','ITUB3.SA','ITUB4.SA','GGBR3.SA','GGBR4.SA']
if not USE_SYNTHETIC:
    _, closes = load_universe(tickers, start='2020-01-01')
    ranking = screen_pairs(closes, window=504)
    display(ranking.round(4))

## Full-sample diagnostics

These estimates describe the sample, not an out-of-sample signal. Use Engle–Granger critical values for fitted residuals; an ordinary residual ADF p-value is not a cointegration test.

In [ ]:
from pairs.stats import adf
ly, lx = np.log(df.close_y), np.log(df.close_x)
alpha, beta = ols(ly, lx)
spread = ly - alpha - beta*lx
stat, p, crit = engle_granger(ly, lx)
ou_ = ou_params(spread)
print(f'alpha={alpha:.4f}  beta={beta:.4f}')
print(f'Engle-Granger: stat={stat:.3f}  p={p:.4f}  critical values (1/5/10%)={np.round(crit,3)}')
print(f'Residual ADF: p={adf(spread)[1]:.4f}  (use the EG p-value for cointegration inference)')
print(f"OU: kappa={ou_['kappa']:.4f}  half-life={ou_['half_life']:.1f} sessions  sigma_eq={ou_['sigma_eq']:.4f}")

## Backtests

Close signals execute at the next open. Commission and slippage apply to each traded leg. Borrow is charged per session on short notional. Remaining positions are liquidated at the final close.

In [ ]:
cfg = BacktestConfig(capital=100_000, commission_bps=3, slippage_bps=5, borrow_bps_annual=200, lot_size=1)
results = {}
for name, fn in STRATEGIES.items():
    sig = fn(df.close_x, df.close_y)
    results[name] = run_backtest(df, sig, cfg)
pd.DataFrame({k: v.summary() for k, v in results.items()}).T.round(3)

In [ ]:
plot_equity(results); plt.show()

In [ ]:
name = 'ou'
plot_signals(df, results[name], name); plt.show()
results[name].trades.tail(10)

## Cost sensitivity

Compare Sharpe across one-way transaction costs. Borrow costs remain fixed.

In [ ]:
sens = {}
for bps in [0, 2.5, 5, 10, 20]:
    c = BacktestConfig(commission_bps=bps/2, slippage_bps=bps/2)
    sens[bps] = {n: run_backtest(df, results[n].signals, c).summary().get('sharpe') for n, f in STRATEGIES.items()}
sens = pd.DataFrame(sens).T
plot_cost_sensitivity(sens); plt.show(); sens.round(2)

## Walk-forward evaluation

Each fold selects parameters from 504 training sessions and evaluates the next 126 sessions. Folds start flat and liquidate at their final close. If no candidate qualifies, the fold stays in cash.

In [ ]:
GRIDS = {
    'relative_return': dict(lookback=[10,20,40], entry=[1.5,2.0,2.5]),
    'cointegration':   dict(window=[126,252], entry=[1.5,2.0,2.5]),
    'ou':              dict(window=[126,252], entry=[1.5,2.0,2.5]),
    'kalman':          dict(delta=[1e-8,1e-7,1e-6], entry=[1.5,2.0,2.5]),
}
wf = {n: walkforward_optimize(df, f, GRIDS[n], cfg) for n, f in STRATEGIES.items()}
display(pd.DataFrame({n: w['summary'] for n, w in wf.items()}).T.round(3))
for n, w in wf.items():
    plt.plot(w['oos_equity'] / w['oos_equity'].iloc[0], label=n)
plt.legend(); plt.title('Out-of-sample equity'); plt.grid(alpha=.3); plt.show()

In [ ]:
wf['ou']['folds']   

## Interpretation

Synthetic results check mechanics; they do not establish a tradable edge. Compare results across pairs and costs, check borrow availability, and keep a final evaluation period untouched. BH assumes independent or suitably dependent tests; pairs sharing assets can violate those assumptions.